Baseline Training Algorithm (COVID vs. Control)
This notebook documents the exact training pipeline implemented in this repository for the first milestone: a lightweight CNN trained on log-mel spectrogram inputs for binary screening.
What this notebook covers:
-Where the training data comes from (splits + feature arrays)
-The preprocessing/feature extraction algorithm (log-mel spectrogram pipeline).
-The label remapping used for the binary milestone
-The baseline CNN architecture
-Training procedure (optimizer, loss, callbacks)
-Evaluation outputs (confusion matrix + classification report)
-Export to TensorFlow Lite (TFLite).
Important note about labels\nThe project’s intended multiclass schema is: TB, COVID, HEALTHY_OR_NONTARGET.
For the first implemented milestone, training is performed as binary screening by remapping the multiclass indices:
`TB (0)` → excluded in the binary milestone
`COVID (1)` → binary class `0`
HEALTHY_OR_NONTARGET (2)` → binary class `1`\n\nThis matches the logic in `src/models/training.py`.

In [ ]:
# Notebook setup: make sure imports from `src/` work.
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()

# If you opened this notebook from a different working directory,
# update PROJECT_ROOT to point at the `thesis-cough-screening/` folder.
if (PROJECT_ROOT / "src").exists() is False and (PROJECT_ROOT / "thesis-cough-screening" / "src").exists():
    PROJECT_ROOT = (PROJECT_ROOT / "thesis-cough-screening").resolve()

sys.path.insert(0, str(PROJECT_ROOT))
print("PROJECT_ROOT:", PROJECT_ROOT)

1) Data inputs used by training\n\nThe training script (`src/models/training.py`) uses one of two sources:
   A. Precomputed feature arrays** (fast path)\n- `data/processed/features/X_train.npy`, `y_train.npy`\n- `data/processed/features/X_val.npy`, `y_val.npy`\n- `data/processed/features/X_test.npy`, `y_test.npy`
   B. Build features on the fly** from split CSVs (slow path)\n- `metadata/splits/train.csv`, `val.csv`, `test.csv`\n- Extract features with the log-mel algorithm defined in `src/features/feature_utils.py`\n\nIn practice, you typically run preprocessing once (via `bash scripts/run_preprocessing.sh`) so that the `.npy` arrays exist.

In [ ]:
import numpy as np

FEATURE_DIR = PROJECT_ROOT / "data" / "processed" / "features"
print("FEATURE_DIR:", FEATURE_DIR)

def load_split_arrays(split_name: str):
    x_path = FEATURE_DIR / f"X_{split_name}.npy"
    y_path = FEATURE_DIR / f"y_{split_name}.npy"
    if not x_path.exists() or not y_path.exists():
        raise FileNotFoundError(
            f"Missing precomputed arrays for '{split_name}'. Expected: {x_path} and {y_path}. "
            "Run `bash scripts/run_preprocessing.sh` first."
        )
    return np.load(x_path), np.load(y_path)

# Load train/val/test if available
# (If these files are not present on this machine, this cell will raise a clear error.)
X_train, y_train = load_split_arrays("train")
X_val, y_val = load_split_arrays("val")
X_test, y_test = load_split_arrays("test")

print("X_train:", X_train.shape, X_train.dtype)
print("y_train:", y_train.shape, y_train.dtype, "unique:", np.unique(y_train))

2. Feature extraction algorithm (log-mel spectrogram)\n\nWhen feature arrays are not precomputed, the pipeline uses the following algorithm (see `src/features/feature_utils.py`):1. Load audio** and standardize sample rate (default 16 kHz) + mono\n2. **Trim silence** (optional, default enabled)\n3. **Extract loudest segment** to focus on cough-like activity\n4. **Peak normalize** (default enabled)\n5. Compute **log-mel spectrogram** with fixed parameters\n6. **Pad/truncate** time frames to `max_frames`\n7. **Min–max scale** to `[0,1]`\n8. Expand to shape `(n_mels, max_frames, 1)`\n\nDefault parameters (from `src/config/settings.py`):\n- `n_mels = 64`\n- `n_fft = 1024`\n- `hop_length = 256`\n- `win_length = 512`\n- `fmin = 50`, `fmax = 8000`\n- `max_frames = 256`\n\nThe result is a fixed-size tensor compatible with a small CNN and mobile export.

3) Label mapping used for the binary milestone\n\nThe training script stores multiclass indices as:\n- `TB = 0`\n- `COVID = 1`\n- `HEALTHY_OR_NONTARGET = 2`\n\nFor the first milestone, training is performed on a binary task by remapping:\n- `COVID (1)` → `0`\n- `HEALTHY_OR_NONTARGET (2)` → `1`\n\nTB (`0`) is excluded from the binary experiment.

In [ ]:
def remap_multiclass_to_binary(y: np.ndarray) -> np.ndarray:
    label_map = {1: 0, 2: 1}  # COVID->0, HEALTHY_OR_NONTARGET->1
    # If TB samples exist in y, this will raise KeyError (by design) to avoid silent leakage.
    return np.array([label_map[int(label)] for label in y], dtype=np.int64)

y_train_bin = remap_multiclass_to_binary(y_train)
y_val_bin = remap_multiclass_to_binary(y_val)
y_test_bin = remap_multiclass_to_binary(y_test)

def summarize_labels(name: str, y: np.ndarray):
    unique, counts = np.unique(y, return_counts=True)
    dist = dict(zip(unique.tolist(), counts.tolist()))
    print(f"{name}: {dist} (total={len(y)})")

summarize_labels("train(binary)", y_train_bin)
summarize_labels("val(binary)", y_val_bin)
summarize_labels("test(binary)", y_test_bin)

4) Baseline CNN architecture\n\nThe baseline CNN is defined in `src/models/baseline_cnn.py` and is intentionally lightweight for mobile deployment.\n\nArchitecture summary:\n- Conv2D(16, 3×3) + BatchNorm + MaxPool\n- Conv2D(32, 3×3) + BatchNorm + MaxPool\n- Conv2D(64, 3×3) + BatchNorm + GlobalAveragePool\n- Dropout(0.3) + Dense(64, ReLU)\n- Dense(num_classes, Softmax)\n\nInput: `(64, 256, 1)` log-mel tensor (default).

In [ ]:
import tensorflow as tf

from src.models.baseline_cnn import build_baseline_cnn

input_shape = X_train.shape[1:]
num_classes = 2
model = build_baseline_cnn(input_shape=input_shape, num_classes=num_classes)
model.summary()

5) Training procedure\n\nThe training procedure implemented in `src/models/training.py` uses:\n- Optimizer: **Adam**\n- Learning rate: `1e-3` (default setting)\n- Loss: `sparse_categorical_crossentropy`\n- Metric: `accuracy`\n- Callbacks:\n  - `ModelCheckpoint` (saves best model)\n  - `EarlyStopping` (patience 5, restores best weights)\n\nDefault training settings (from `src/config/settings.py`):\n- `batch_size = 16`\n- `epochs = 15` (upper bound; early stopping can stop earlier)\n- `validation_monitor = val_loss`\n\nBelow, we reproduce the same logic in-notebook.

In [ ]:
from src.config.settings import load_settings
from src.utils.seed import set_seed

settings = load_settings()
set_seed(settings.splits.random_state)

optimizer = tf.keras.optimizers.Adam(learning_rate=settings.training.learning_rate)
model.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

CHECKPOINT_DIR = PROJECT_ROOT / "models" / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

checkpoint_path = CHECKPOINT_DIR / "baseline_cnn_notebook.keras"
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_path,
        monitor=settings.training.validation_monitor,
        save_best_only=True,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor=settings.training.validation_monitor,
        patience=settings.training.early_stopping_patience,
        restore_best_weights=True,
    ),
]

history = model.fit(
    X_train,
    y_train_bin,
    validation_data=(X_val, y_val_bin),
    batch_size=settings.training.batch_size,
    epochs=settings.training.epochs,
    callbacks=callbacks,
    verbose=1,
)

print("Saved checkpoint:", checkpoint_path)

6) Evaluation (classification report + confusion matrix)\n\nFor screening tasks, **class-wise recall** is particularly important because false negatives (missed disease) can be costly. The evaluation below reports both overall accuracy and the confusion matrix.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

test_loss, test_acc = model.evaluate(X_test, y_test_bin, verbose=1)
probs = model.predict(X_test, verbose=0)
pred = np.argmax(probs, axis=1)

print(f"Test accuracy: {test_acc*100:.2f}%")
print(classification_report(y_test_bin, pred, target_names=["COVID", "CONTROL"]))
print("Confusion matrix [rows=true, cols=pred]:")
print(confusion_matrix(y_test_bin, pred))

7) Export to TensorFlow Lite (TFLite)\n\nThe repository includes a simple export utility in `src/export/export_tflite.py`. In some environments, direct conversion can fail due to converter/operator issues; a known workaround is to export via a concrete function (and allow select TF ops).\n\nBelow is the **standard export path** used by the project. If you encounter conversion failures, use the workaround shown afterwards.

In [ ]:
from src.export.export_tflite import export_tflite

EXPORTED_DIR = PROJECT_ROOT / "models" / "exported"
EXPORTED_DIR.mkdir(parents=True, exist_ok=True)
tflite_path = EXPORTED_DIR / "baseline_cnn_notebook.tflite"

# Export the best checkpoint saved during this notebook run
export_tflite(checkpoint=checkpoint_path, output=tflite_path)
print("Exported:", tflite_path, "size(bytes)=", tflite_path.stat().st_size)

Optional workaround (if conversion crashes in your environment)\n\nIf `from_keras_model` conversion crashes, this concrete-function method is a robust fallback (it is also the approach used during the RunPod export when the converter crashed on batch normalization graphs).

In [ ]:
def export_tflite_concrete_function(model: tf.keras.Model, output_path: Path) -> Path:
    input_spec = tf.TensorSpec(shape=[None, *model.input_shape[1:]], dtype=tf.float32)

    @tf.function(input_signature=[input_spec])
    def predict_fn(x):
        return model(x, training=False)

    concrete = predict_fn.get_concrete_function()
    converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete])
    converter.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS,
        tf.lite.OpsSet.SELECT_TF_OPS,
    ]
    tflite_model = converter.convert()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_bytes(tflite_model)
    return output_path

# Example:
# export_tflite_concrete_function(model, EXPORTED_DIR / "baseline_cnn_concrete.tflite")

Supervisor-facing summary (one paragraph)\n\nThis project implements a reproducible cough-audio screening pipeline: public datasets are audited and unified under a controlled label schema, cough recordings are standardized to 16 kHz mono and transformed into fixed-shape log-mel spectrogram tensors (64 mel bins × 256 frames × 1 channel), and a lightweight CNN is trained using Adam optimization with early stopping and checkpointing. For the first milestone, labels are remapped to a binary screening task (COVID vs. control), allowing mobile-oriented model export to TensorFlow Lite for later app integration and device-level benchmarking.